# Compare old trainer data vs Lightning DataModule

This notebook compares data preparation between the legacy pipeline ([src/trainer.py](../src/trainer.py)) and the Lightning pipeline ([src/lightning_data.py](../src/lightning_data.py)).

In [1]:
import os
import sys
import yaml
import numpy as np
import torch

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import data_util
from lightning_data import CaloINNDataModule

torch.set_default_dtype(torch.float32)
np.random.seed(0)
torch.manual_seed(0)

print('repo_root =', repo_root)

repo_root = /global/cfs/cdirs/m3443/usr/pmtuan/caloinn_lightning


In [2]:
old_cfg_path = os.path.join(repo_root, 'params', 'pions.yaml')
new_cfg_path = os.path.join(repo_root, 'params', 'lightning_pion.yaml')

with open(old_cfg_path, 'r') as f:
    old_cfg = yaml.safe_load(f)
with open(new_cfg_path, 'r') as f:
    new_cfg = yaml.safe_load(f)

new_data = new_cfg['data']
new_ds_kwargs = new_data['dataset_kwargs']
new_model = new_cfg['model']

mapping = {
    'data_path': (old_cfg.get('data_path'), new_data.get('data_path')),
    'val_data_path': (old_cfg.get('val_data_path'), new_data.get('val_data_path')),
    'batch_size': (old_cfg.get('batch_size'), new_data.get('batch_size')),
    'val_frac': (old_cfg.get('val_frac'), new_data.get('val_frac')),
    'xml_path': (old_cfg.get('xml_path'), new_ds_kwargs.get('xml_path')),
    'xml_ptype': (old_cfg.get('xml_ptype'), new_ds_kwargs.get('xml_ptype')),
    'single_energy': (old_cfg.get('single_energy'), new_ds_kwargs.get('single_energy')),
    'eps': (old_cfg.get('eps'), new_ds_kwargs.get('eps')),
    'u0up_cut': (old_cfg.get('u0up_cut'), new_ds_kwargs.get('u0up_cut')),
    'u0low_cut': (old_cfg.get('u0low_cut'), new_ds_kwargs.get('u0low_cut')),
    'pt_rew': (old_cfg.get('pt_rew'), new_ds_kwargs.get('pt_rew')),
    'dep_cut': (old_cfg.get('dep_cut'), new_ds_kwargs.get('dep_cut')),
    'width_noise': (old_cfg.get('width_noise'), new_model.get('width_noise')),
    'lr': (old_cfg.get('lr'), new_model.get('optimizer_params', {}).get('lr')),
    'max_lr': (old_cfg.get('max_lr'), new_model.get('scheduler_params', {}).get('max_lr')),
    'weight_decay': (old_cfg.get('weight_decay'), new_model.get('optimizer_params', {}).get('weight_decay')),
    'betas': (old_cfg.get('betas'), new_model.get('optimizer_params', {}).get('betas')),
    'n_epochs': (old_cfg.get('n_epochs'), new_model.get('scheduler_params', {}).get('epochs')),
}

print('Config compatibility check (old vs lightning):')
mismatches = []
for k, (v_old, v_new) in mapping.items():
    ok = v_old == v_new
    print(f'- {k}: {v_old} | {v_new} | match={ok}')
    if not ok:
        mismatches.append(k)

print('\nMismatches:', mismatches if mismatches else 'None')

Config compatibility check (old vs lightning):
- data_path: /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | match=True
- val_data_path: /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | match=True
- batch_size: 512 | 512 | match=True
- val_frac: 0.01 | 0.01 | match=True
- xml_path: /global/cfs/cdirs/m3443/usr/pmtuan/caloinn_lightning/binning_dataset_1_pions.xml | /global/cfs/cdirs/m3443/usr/pmtuan/caloinn_lightning/binning_dataset_1_pions.xml | match=True
- xml_ptype: pion | pion | match=True
- single_energy: None | None | match=True
- eps: 1e-10 | 1e-10 | match=True
- u0up_cut: 3.5 | 3.5 | match=True
- u0low_cut: 0.0 | 0.0 | match=True
- pt_rew: 1.0 | 1.0 | match=True
- dep_cut: 600 | 600 | match=True
- width_noise: 5e-06 | 5e-06 | match=True
- lr: 1e-05 | 1e-05 | match=True
- max_lr: 0.0001 | 0.0001 |

## 1) Same raw index range -> same prepared tensors?

This directly validates whether the new dataset class applies the same `load_data` + `preprocess` logic as the legacy pipeline when given the same raw events.

In [3]:
seed = 2026
torch.manual_seed(seed)
np.random.seed(seed)

# Legacy reference loader
old_train_loader, _, _ = data_util.get_loaders(
    old_cfg.get('data_path'),
    old_cfg.get('xml_path'),
    old_cfg.get('xml_ptype'),
    old_cfg.get('val_frac'),
    old_cfg.get('batch_size'),
    old_cfg.get('eps', 1.0e-10),
    device='cpu',
    shuffle=old_cfg.get('shuffle', False),
    width_noise=old_cfg.get('width_noise', 1e-7),
    energy=old_cfg.get('single_energy', None),
    u0up_cut=old_cfg.get('u0up_cut', 7.0),
    u0low_cut=old_cfg.get('u0low_cut', 0.0),
    rew=old_cfg.get('pt_rew', 1.0),
    dep_cut=old_cfg.get('dep_cut', 1.0e10),
)

# Lightning DataModule now reuses the same legacy loading path in setup('fit')
torch.manual_seed(seed)
np.random.seed(seed)
dm = CaloINNDataModule(**new_data)
dm.setup('fit')
new_train_loader = dm.train_dataloader()

# Reseed before each iterator creation to compare exact same random permutation/noise draws
torch.manual_seed(seed)
np.random.seed(seed)
old_x, old_c = next(iter(old_train_loader))

torch.manual_seed(seed)
np.random.seed(seed)
new_x, new_c = next(iter(new_train_loader))

print('old shape x,c:', tuple(old_x.shape), tuple(old_c.shape))
print('new shape x,c:', tuple(new_x.shape), tuple(new_c.shape))

max_abs_x = float(torch.max(torch.abs(old_x - new_x)).item())
max_abs_c = float(torch.max(torch.abs(old_c - new_c)).item())
print('max_abs_diff x:', max_abs_x)
print('max_abs_diff c:', max_abs_c)
print('allclose x (atol=1e-7):', bool(torch.allclose(old_x, new_x, atol=1e-7, rtol=0.0)))
print('allclose c (atol=1e-12):', bool(torch.allclose(old_c, new_c, atol=1e-12, rtol=0.0)))

print('num_train_samples old/new:', len(old_train_loader.data), dm.num_train_samples)

old shape x,c: (512, 540) (512, 1)
new shape x,c: (512, 540) (512, 1)
max_abs_diff x: 5.0067901611328125e-06
max_abs_diff c: 0.0
allclose x (atol=1e-7): False
allclose c (atol=1e-12): True
num_train_samples old/new: 119516 119516


## 2) Compare first training batch (old trainer vs Lightning datamodule)

This checks what enters each training loop path. Note: old `MyDataLoader` adds noise in the loader, while Lightning adds noise in `training_step`.

In [6]:
# Compare the exact first training batch tensors before any stochastic noise is applied.
batch_size = int(old_cfg.get('batch_size'))

old_train_batch_x = old_train_loader.data[:batch_size]
old_train_batch_c = old_train_loader.cond[:batch_size]
new_train_batch_x = dm._train_dataset.tensors[0][:batch_size]
new_train_batch_c = dm._train_dataset.tensors[1][:batch_size]

print('old first batch x,c:', tuple(old_train_batch_x.shape), tuple(old_train_batch_c.shape))
print('new first batch x,c:', tuple(new_train_batch_x.shape), tuple(new_train_batch_c.shape))
print('batch x max_abs_diff:', float(torch.max(torch.abs(old_train_batch_x - new_train_batch_x)).item()))
print('batch c max_abs_diff:', float(torch.max(torch.abs(old_train_batch_c - new_train_batch_c)).item()))
print('batch x allclose    :', bool(torch.allclose(old_train_batch_x, new_train_batch_x, rtol=0.0, atol=1e-7)))
print('batch c allclose    :', bool(torch.allclose(old_train_batch_c, new_train_batch_c, rtol=0.0, atol=1e-12)))

old first batch x,c: (512, 540) (512, 1)
new first batch x,c: (512, 540) (512, 1)
batch x max_abs_diff: 0.0
batch c max_abs_diff: 0.0
batch x allclose    : True
batch c allclose    : True


## 3) Compare train/val split semantics (full loaders, no noise)

This section compares how samples are assigned to train vs val in the old pipeline and in the Lightning datamodule.

Both sides use `shuffle=False` and `width_noise=0.0` to isolate split logic only.

In [7]:
# Old pipeline split (after preprocess), no noise.
old_train_loader, old_val_loader, _ = data_util.get_loaders(
    old_cfg.get('data_path'),
    xml_abs,
    old_cfg.get('xml_ptype'),
    old_cfg.get('val_frac'),
    old_cfg.get('batch_size'),
    old_cfg.get('eps'),
    device='cpu',
    width_noise=0.0,
    energy=old_cfg.get('single_energy', None),
    u0up_cut=old_cfg.get('u0up_cut', 7.0),
    u0low_cut=old_cfg.get('u0low_cut', 0.0),
    rew=old_cfg.get('pt_rew', 1.0),
    dep_cut=old_cfg.get('dep_cut', 1e10),
    shuffle=False,
)

# Lightning split from the same raw input, no noise.
dm_split = CaloINNDataModule(
    data_path=new_data['data_path'],
    val_data_path=new_data['val_data_path'],
    batch_size=new_data['batch_size'],
    cond_key=new_data.get('cond_key', 'incident_energies'),
    sample_key=new_data.get('sample_key', 'showers'),
    val_frac=new_data.get('val_frac', 0.01),
    shuffle=False,
    eval_dataset=new_data.get('eval_dataset', '1-pions'),
    num_workers=0,
    predict_batch_size=new_data.get('predict_batch_size', 1000),
    dataset_kwargs=new_ds_kwargs_local,
)
dm_split.setup('fit')

old_train_x_all = torch.clone(old_train_loader.data).cpu()
old_train_c_all = torch.clone(old_train_loader.cond).cpu()
old_val_x_all = torch.clone(old_val_loader.data).cpu()
old_val_c_all = torch.clone(old_val_loader.cond).cpu()

new_train_x_all = torch.clone(dm_split._train_dataset.tensors[0]).cpu()
new_train_c_all = torch.clone(dm_split._train_dataset.tensors[1]).cpu()
new_val_x_all = torch.clone(dm_split._val_dataset.tensors[0]).cpu()
new_val_c_all = torch.clone(dm_split._val_dataset.tensors[1]).cpu()

print('Counts:')
print(f"- old train: {old_train_x_all.shape[0]}")
print(f"- old val  : {old_val_x_all.shape[0]}")
print(f"- new train: {new_train_x_all.shape[0]}")
print(f"- new val  : {new_val_x_all.shape[0]}")

print('\nDirect tensor parity:')
print('- train x max_abs_diff:', float(torch.max(torch.abs(old_train_x_all - new_train_x_all)).item()))
print('- train c max_abs_diff:', float(torch.max(torch.abs(old_train_c_all - new_train_c_all)).item()))
print('- val x max_abs_diff  :', float(torch.max(torch.abs(old_val_x_all - new_val_x_all)).item()))
print('- val c max_abs_diff  :', float(torch.max(torch.abs(old_val_c_all - new_val_c_all)).item()))
print('- train x allclose    :', bool(torch.allclose(old_train_x_all, new_train_x_all, rtol=0.0, atol=1e-7)))
print('- train c allclose    :', bool(torch.allclose(old_train_c_all, new_train_c_all, rtol=0.0, atol=1e-12)))
print('- val x allclose      :', bool(torch.allclose(old_val_x_all, new_val_x_all, rtol=0.0, atol=1e-7)))
print('- val c allclose      :', bool(torch.allclose(old_val_c_all, new_val_c_all, rtol=0.0, atol=1e-12)))

Counts:
- old train: 119516
- old val  : 1207
- new train: 119516
- new val  : 1207

Direct tensor parity:
- train x max_abs_diff: 0.0
- train c max_abs_diff: 0.0
- val x max_abs_diff  : 0.0
- val c max_abs_diff  : 0.0
- train x allclose    : True
- train c allclose    : True
- val x allclose      : True
- val c allclose      : True


## 4) Summary helper
Run this cell after the previous ones for a compact verdict.

In [8]:
print('Section 1 should report exact tensor parity for train and val splits.')
print('Section 2 should report exact first-batch tensor parity when comparing the raw tensors directly.')
print('Practical fit parity is still the final check for training behavior.')

Section 1 should report exact tensor parity for train and val splits.
Section 2 should report exact first-batch tensor parity when comparing the raw tensors directly.
Practical fit parity is still the final check for training behavior.


In [5]:
# Direct notebook verdict: compare raw tensors between legacy and Lightning data prep.
seed = 2026
np.random.seed(seed)
torch.manual_seed(seed)

xml_path = old_cfg.get('xml_path')
xml_abs = xml_path
if xml_abs.startswith('./'):
    xml_abs = os.path.join(repo_root, xml_abs[2:])
xml_abs = os.path.abspath(xml_abs)

batch_size = int(old_cfg.get('batch_size'))
new_ds_kwargs_local = dict(new_ds_kwargs)
new_ds_kwargs_local['xml_path'] = xml_abs

old_train_loader, old_val_loader, _ = data_util.get_loaders(
    old_cfg.get('data_path'),
    xml_abs,
    old_cfg.get('xml_ptype'),
    old_cfg.get('val_frac'),
    old_cfg.get('batch_size'),
    old_cfg.get('eps'),
    device='cpu',
    width_noise=0.0,
    energy=old_cfg.get('single_energy', None),
    u0up_cut=old_cfg.get('u0up_cut', 7.0),
    u0low_cut=old_cfg.get('u0low_cut', 0.0),
    rew=old_cfg.get('pt_rew', 1.0),
    dep_cut=old_cfg.get('dep_cut', 1e10),
    shuffle=False,
)

dm_check = CaloINNDataModule(
    data_path=new_data['data_path'],
    val_data_path=new_data['val_data_path'],
    batch_size=new_data['batch_size'],
    cond_key=new_data.get('cond_key', 'incident_energies'),
    sample_key=new_data.get('sample_key', 'showers'),
    val_frac=new_data.get('val_frac', 0.01),
    shuffle=False,
    eval_dataset=new_data.get('eval_dataset', '1-pions'),
    num_workers=0,
    predict_batch_size=new_data.get('predict_batch_size', 1000),
    dataset_kwargs=new_ds_kwargs_local,
)
dm_check.setup('fit')

train_x_diff = float(torch.max(torch.abs(old_train_loader.data - dm_check._train_dataset.tensors[0])).item())
train_c_diff = float(torch.max(torch.abs(old_train_loader.cond - dm_check._train_dataset.tensors[1])).item())
val_x_diff = float(torch.max(torch.abs(old_val_loader.data - dm_check._val_dataset.tensors[0])).item())
val_c_diff = float(torch.max(torch.abs(old_val_loader.cond - dm_check._val_dataset.tensors[1])).item())

print('train x diff:', train_x_diff)
print('train c diff:', train_c_diff)
print('val x diff  :', val_x_diff)
print('val c diff  :', val_c_diff)
print('train exact :', train_x_diff == 0.0 and train_c_diff == 0.0)
print('val exact   :', val_x_diff == 0.0 and val_c_diff == 0.0)

first_batch_x_diff = float(torch.max(torch.abs(old_train_loader.data[:batch_size] - dm_check._train_dataset.tensors[0][:batch_size])).item())
first_batch_c_diff = float(torch.max(torch.abs(old_train_loader.cond[:batch_size] - dm_check._train_dataset.tensors[1][:batch_size])).item())
print('first batch x diff:', first_batch_x_diff)
print('first batch c diff:', first_batch_c_diff)

train x diff: 0.0
train c diff: 0.0
val x diff  : 0.0
val c diff  : 0.0
train exact : True
val exact   : True
first batch x diff: 0.0
first batch c diff: 0.0
